|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 2. You can batch the requests, and you can schedule them
once for each iteration. Now the failures come from many users at the same
time: padding, masks, queues and batch sizes.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 05. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 2.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | Memory | Bandwidth | bf16 compute |
|---|---|---|---|
| A100 SXM 80GB | 80 GB | 2,039 GB/s | 312 TFLOP/s |
| L40S | 48 GB | 864 GB/s | 362 TFLOP/s |

| Model | Layers | KV heads | head_dim | bf16 weights | KV bytes per token |
|---|---|---|---|---|---|
| Qwen3-1.7B | 28 | 8 | 128 | 3.44 GB | 114,688 (112 KiB) |
| Llama-3-8B | 32 | 8 | 128 | 16.1 GB | 131,072 (128 KiB) |

Three formulas from Parts 1 and 2:

    decode step (s)  >= (weight bytes + B x context x KV bytes per token) / bandwidth
    decode FLOP      =  2 x parameters x B
    Little's law     :  requests in the system = arrival rate x time in the system

# Ticket 1: the short prompts get garbage

**Severity:** high. **Reported by:** the team that moved to batching.

> We batch 8 prompts now. The longest prompt of each batch gets a good
> answer. The other prompts get garbage. When we run them one at a time,
> all of them are good.

**Evidence**

- The batching code:

  ```python
  batch = tokenizer(prompts, padding=True, return_tensors='pt')
  out = model(batch.input_ids, attention_mask=batch.attention_mask, use_cache=True)
  next_tokens = out.logits[:, -1].argmax(-1)
  ```

- `print(tokenizer.padding_side)` prints `right`.
- A sample batch: the prompt lengths are 31, 24, 31, 12, 18, 31, 27, 9.
  The three prompts of length 31 get good answers.
- The GPU ran out of memory one time last week, at batch 16.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the first token is always right

**Severity:** medium. **Reported by:** the evaluation team.

> After the fix for the right padding, the batched answers look fluent.
> But the benchmark score in batch mode is 4 points lower than one at a
> time.

**Evidence**

- The code pads on the left now. The prefill passes the mask. The decode
  loop:

  ```python
  out = model(batch.input_ids, attention_mask=batch.attention_mask, use_cache=True)
  cache = out.past_key_values
  next_tokens = out.logits[:, -1:].argmax(-1)
  for _ in range(max_tokens - 1):
      out = model(next_tokens, past_key_values=cache, use_cache=True)
      cache = out.past_key_values
      next_tokens = out.logits[:, -1:].argmax(-1)
  ```

- The team compared each row of a batch with the same prompt alone, in
  fp32:

  | row | pads | first different token |
  |---|---|---|
  | 0 | 0 | none |
  | 1 | 3 | 2 |
  | 2 | 11 | 1 |
  | 3 | 0 | none |
  | 4 | 19 | 1 |

- The team thinks that this is the bf16 batch effect of Part 1.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: batch 256 is only 19% faster than batch 128

**Severity:** low. **Reported by:** the performance team.

> From batch 1 to batch 16, the throughput grew almost 16x. From batch
> 128 to batch 256 it grew only 1.19x. The scheduler must add overhead
> at a large batch.

**Evidence**

- Qwen3-1.7B on an L40S. Each sequence has about 512 tokens of context
  during the test.
- The measured decode throughput:

  | batch | tokens/s |
  |---|---|
  | 64 | 6,100 |
  | 128 | 8,050 |
  | 256 | 9,560 |

- The profiler: 92% of each step at batch 256 is GPU kernels.
- A colleague says: "At batch 256 the card is compute-bound. That is the
  roofline."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: the job that took nine times longer

**Severity:** medium. **Reported by:** the data team.

> We summarize 10,000 support tickets each night with a static batch of
> 32. The plan said 3.5 hours. It took 31 hours. The GPU was at 100%
> utilization the whole time.

**Evidence**

- The plan: each batch runs for the mean output length, 176 tokens.
- The output lengths: 96% of the summaries have about 100 tokens. 4% are
  long tickets, and their summaries run to the limit of 2,000 tokens.
- Without the long ones, the longest summary in a batch of 32 is about
  300 tokens.
- The code is the static batch of stage 04: a batch ends when its last
  sequence ends.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: continuous batching that barely helps

**Severity:** medium. **Reported by:** the platform team.

> We replaced the static batch with our continuous batching scheduler.
> The paper promises 2x to 4x. We get 1.3x. The waiting queue is always
> long, so we need more GPUs.

**Evidence**

- The scheduler log shows `running=32` at every step of the peak hour.
- The end of the step:

  ```python
  for seq in self.running:
      seq.output.append(next_tokens[seq.slot])
  self.running = [s for s in self.running
                  if len(s.output) < s.max_tokens]
  ```

- The answers that the users receive are correct. The [detokenizer](../../GLOSSARY.md#detokenization) stops
  the text at the first `<|im_end|>`.
- `max_tokens` is 1,024. The [median](../../GLOSSARY.md#percentile) answer has 180 tokens.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the afternoon that gets slower and slower

**Severity:** high. **Reported by:** users.

> From 13:00 the first token takes longer and longer. At 13:30 it takes
> 6 minutes. At 14:00 it takes 12 minutes. A restart fixes it for a
> short time.

**Evidence**

- The server finishes 10 requests each second when it is full. Stage 05
  measured this.
- The arrival rate from 13:00 to 15:00 is 12 requests each second. The
  rest of the day it is 6.
- The time for each output token stays at 25 ms all afternoon.
- The KV cache is 96% full from 13:00.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: the bot forgets who said what

**Severity:** medium. **Reported by:** customer support.

> In long conversations the bot mixes up the turns. It answers questions
> that the user asked three turns ago, or it thinks that it said things
> that the user said. Single questions work.

**Evidence**

- The model is Llama-3-8B-Instruct. Its tokenizer has no pad token, so
  the code sets `tokenizer.pad_token = tokenizer.eos_token`, which is
  `<|eot_id|>` (id 128009).
- The chat template ends each turn with `<|eot_id|>`.
- The batching code builds the mask itself:

  ```python
  ids = pad_left(token_lists, pad_id=tokenizer.pad_token_id)
  mask = (ids != tokenizer.pad_token_id).long()
  ```

- The path for a single request does not build a mask.
- The team thinks that the model is weak in long conversations.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: the fastest batch size makes users unhappy

**Severity:** medium. **Reported by:** the product team.

> We set the maximum batch to 192 because it gave the most tokens per
> second. Now users say that the text appears too slowly. The product
> promise is at least 20 tokens each second for each user.

**Evidence**

- Llama-3-8B on an A100. The measured decode steps:

  | batch | ms for each step |
  |---|---|
  | 64 | 22 |
  | 128 | 38 |
  | 192 | 55 |
  | 256 | 71 |

- The GPU utilization is 100% at 192. The team says that this is
  efficient.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**